# **Inferencia: HybridCNN Audio Classifier (`multiclase`)**

Carga el modelo entrenado y predice sobre audios `.wav` / `.mp3` usando **exactamente el mismo pipeline** que `preprocess.py → main()` (sr=44100, n_mels=128, n_mfcc=13, n_fft=2048, hop=512, sin `target_duration`).

## 1 · Importaciones

In [34]:
from __future__ import annotations

import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

warnings.filterwarnings("ignore")

from src.utils.config import *
from src.models.hybrid_cnn_v3 import ImprovedMFCCCNN
from src.data.preprocess import Preprocess, PreprocessConfig  # ← pipeline centralizado

print("✅ Importaciones completadas")


✅ Importaciones completadas


## 2 · Configuración

In [35]:
# ─────────────────────────────────────────────
# 🔧 RUTAS
# ─────────────────────────────────────────────
MODEL_PATH         = FINAL_MODEL_DIR / "best_human_label_v6.pt"
# MODEL_PATH       = CHECKPOINT_DIR /"human_label"/ "checkpoint_epoch_22.pt"   # ← descomenta si usas checkpoint
# MODEL_PATH       = CHECKPOINT_DIR /"human_label_V3"/ "checkpoint_epoch_24.pt"   # ← descomenta si usas checkpoint
# MODEL_PATH       = FINAL_MODEL_DIR / "model_total_V4_32.pt"   # ← descomenta si usas checkpoint

LABEL_MAPPING_PATH = LABEL_MAPPING["human_label2"]

AUDIO_FOLDER       = TEST_AUDIO_FOLDER / "alertables"   # carpeta con .wav / .mp3

# # ─────────────────────────────────────────────
# # 🎛️ PARÁMETROS DE AUDIO  (igual que preprocess.py → main())
# # ─────────────────────────────────────────────
# SAMPLE_RATE    = 44100
# N_MELS         = 128
# N_MFCC         = 13      # igual que preprocess
# N_FFT          = 2048
# HOP_LENGTH     = 512
# PEAK_TARGET    = 0.99    # normalización de pico
# ─────────────────────────────────────────────
# 🎛️ PARÁMETROS DE AUDIO  (igual que preprocess.py → main())
# ─────────────────────────────────────────────
SAMPLE_RATE    = 16000
N_MELS         = 128
N_MFCC         = 13
N_FFT          = 1024
HOP_LENGTH     = 160
PEAK_TARGET    = 0.99
# ─────────────────────────────────────────────
# 🧠 MODELO
# ─────────────────────────────────────────────
MODE           = "mel_waveform"   # "mel_only" | "mfcc_only" | "mel_mfcc" | "mel_waveform"
DROPOUT        = 0.25
AUGMENT_INFERENCE=False
# ─────────────────────────────────────────────
# 🖥️ INFERENCIA
# ─────────────────────────────────────────────
TOP_K          = 5       # cuántas predicciones mostrar por audio

print(f"📂 Carpeta de audios : {AUDIO_FOLDER}")
print(f"📦 Modelo            : {MODEL_PATH}")
print(f"🗂️  Label mapping      : {LABEL_MAPPING_PATH}")

📂 Carpeta de audios : /home/andres/Documentos/proyecto4geeks/tests/audios/alertables
📦 Modelo            : /home/andres/Documentos/proyecto4geeks/models/final/best_human_label_v6.pt
🗂️  Label mapping      : /home/andres/Documentos/proyecto4geeks/data/interim/processed_dataset/label_mapping_human_labelV2.pkl


## 3 · Cargar label mapping

In [36]:
def load_label_mapping(path: Path) -> tuple[dict, dict, int]:
    """Devuelve (label2idx, idx2label, num_classes)."""
    with open(path, "rb") as f:
        payload = pickle.load(f)

    if isinstance(payload, dict) and "label2idx" in payload:
        label2idx = payload["label2idx"]
        idx2label = payload["idx2label"]
    else:
        label2idx = payload
        idx2label = {v: k for k, v in label2idx.items()}

    num_classes = len(label2idx)
    return label2idx, idx2label, num_classes


label2idx, idx2label, NUM_CLASSES = load_label_mapping(LABEL_MAPPING_PATH)

print(f"✅ {NUM_CLASSES} clases cargadas")
print("   Clases:", list(label2idx.keys()))

✅ 10 clases cargadas
   Clases: ['car_crash', 'construction_noise', 'crying', 'dog', 'fight', 'fire', 'glass_breaking', 'gun_explosion', 'siren_alarm', 'traffic']


## 4 · Instanciar `Preprocess` (pipeline centralizado)

En lugar de construir los transforms manualmente, se delega en `Preprocess`.
Los parámetros deben ser **idénticos** a los usados en `preprocess.py → main()`.


In [37]:
# Instanciar Preprocess con los mismos parámetros que preprocess.py → main()
# target_duration=None → sin recorte (comportamiento de inferencia)
pp_config = PreprocessConfig(
    sample_rate=SAMPLE_RATE,
    target_duration=None,       # sin padding/recorte en inferencia
    normalize_peak=True,
    peak_target=PEAK_TARGET,
    n_mels=N_MELS,
    n_mfcc=N_MFCC,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    save_audio=False,           # no guardar nada en inferencia
    save_mel=False,
    save_mfcc=False,
)

pp = Preprocess(config=pp_config)
device = pp.device

print(f"🖥️  Dispositivo: {device}")
print("✅ Preprocess instanciado — transforms listos")


🖥️  Dispositivo: cuda
✅ Preprocess instanciado — transforms listos


## 5 · Pipeline de preprocesado — delegado en `Preprocess.process_audio_file()`

`Preprocess.process_audio_file(path)` aplica exactamente el mismo pipeline
que `preprocess.py → main()`: mono → resampleo → fix_length → normalización de pico → mel + mfcc.


In [38]:
def load_and_preprocess(
    audio_path: Path,
    augment_inference: bool = AUGMENT_INFERENCE,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Wrapper que delega en Preprocess.process_audio_file().
    Devuelve (mel, mfcc, waveform) con shape:
      mel      : [1, N_MELS, T]  float32
      mfcc     : [1, N_MFCC, T]  float32
      waveform : [1, T]           float32  — necesario para modo mel_waveform
    """
    mel, mfcc = pp.process_audio_file(audio_path, augment_inference=augment_inference)
    # Obtener waveform por separado para mel_waveform
    # process_audio_file ya aplica todo el pipeline; recalculamos solo el waveform
    import soundfile as sf
    import torchaudio.transforms as T

    audio_np, sr = sf.read(str(audio_path))
    waveform = torch.tensor(audio_np, dtype=torch.float32)
    if waveform.ndim == 1:
        waveform = waveform.unsqueeze(0)
    else:
        waveform = waveform.transpose(0, 1)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    waveform = waveform.to(device)
    if sr != SAMPLE_RATE:
        resampler = T.Resample(orig_freq=sr, new_freq=SAMPLE_RATE).to(device)
        waveform = resampler(waveform)
    peak = waveform.abs().max().clamp_min(1e-8)
    waveform = (waveform / peak * PEAK_TARGET).float()

    return mel.float(), mfcc.float(), waveform


print("✅ load_and_preprocess() listo (delega en Preprocess)")

✅ load_and_preprocess() listo (delega en Preprocess)


## 6 · Cargar modelo

In [39]:
def detect_mode_from_state_dict(state_dict: dict) -> str:
    keys = set(state_dict.keys())
    has_mel      = any(k.startswith("cnn_mel.")      for k in keys)
    has_mfcc     = any(k.startswith("cnn_mfcc.")     for k in keys)
    has_waveform = any(k.startswith("cnn_wave.") for k in keys)

    if has_mel and has_waveform:
        return "mel_waveform"
    elif has_mel and has_mfcc:
        return "mel_mfcc"
    elif has_mel:
        return "mel_only"
    elif has_mfcc:
        return "mfcc_only"
    else:
        raise ValueError("No se encontraron keys cnn_mel.*, cnn_mfcc.* ni waveform_cnn.* en el state_dict.")


def load_model(model_path: Path, num_classes: int, dropout: float) -> tuple[ImprovedMFCCCNN, str]:
    checkpoint = torch.load(model_path, map_location=device)

    if isinstance(checkpoint, dict) and "model_state" in checkpoint:
        state_dict = checkpoint["model_state"]
        epoch_info = checkpoint.get("epoch", "?")
        acc_info   = checkpoint.get("best_acc", "?")
        print(f"   📌 Checkpoint — epoch: {epoch_info}  |  best_acc: {acc_info}")
    else:
        state_dict = checkpoint

    detected_mode = detect_mode_from_state_dict(state_dict)
    print(f"   🔍 Modo detectado automáticamente: {detected_mode}")

    model = ImprovedMFCCCNN(num_classes=num_classes, dropout=dropout, mode=detected_mode).to(device)
    model.load_state_dict(state_dict)
    model.eval()
    return model, detected_mode


model, MODE = load_model(MODEL_PATH, NUM_CLASSES, DROPOUT)

total_params = sum(p.numel() for p in model.parameters())
print(f"✅ Modelo cargado  |  {total_params:,} parámetros  |  modo: {MODE}")

   🔍 Modo detectado automáticamente: mel_waveform
✅ Modelo cargado  |  1,842,580 parámetros  |  modo: mel_waveform


## 7 · Función de predicción

In [40]:
@torch.inference_mode()
def predict(audio_path: Path, top_k: int = TOP_K) -> dict:
    """
    Retorna un dict con:
      - filename    : nombre del archivo
      - prediction  : clase predicha (bool — True=alertable, False=no alertable)
      - confidence  : probabilidad de la clase predicha (float)
      - top_k       : lista de (clase, prob) para las top_k predicciones
      - logits_raw  : tensor de logits (para debugging)
    """
    mel, mfcc, waveform = load_and_preprocess(audio_path)

    # Añadir dimensión de batch → [1, 1, F, T] / [1, 1, T]
    mel_b      = mel.unsqueeze(0)
    mfcc_b     = mfcc.unsqueeze(0)
    waveform_b = waveform.unsqueeze(0)

    if MODE == "mel_only":
        logits = model(mel=mel_b)
    elif MODE == "mel_mfcc":
        logits = model(mel=mel_b, mfcc=mfcc_b)
    elif MODE == "mel_waveform":
        logits = model(mel=mel_b, waveform=waveform_b)
    else:
        raise ValueError(f"Modo desconocido: {MODE}")

    probs = torch.softmax(logits, dim=-1).squeeze(0).cpu()

    top_probs, top_idxs = probs.topk(min(top_k, len(idx2label)))

    pred_idx   = int(top_idxs[0])
    pred_label = idx2label[pred_idx]
    pred_conf  = float(top_probs[0])

    top_list = [(idx2label[int(i)], float(p)) for i, p in zip(top_idxs, top_probs)]

    return {
        "filename"   : audio_path.name,
        "prediction" : pred_label,
        "confidence" : pred_conf,
        "top_k"      : top_list,
        "logits_raw" : logits.squeeze(0).cpu(),
    }


print("✅ Función predict() lista")

✅ Función predict() lista


## 8 · Probar un audio individual (opcional)

In [41]:
# ── Cambia esto al archivo que quieras probar ───────────────────────────
# SINGLE_AUDIO = Path("/ruta/al/audio.wav")
# ────────────────────────────────────────────────────────────────────────

# Demo: coge el primer audio de la carpeta si existe
audios = sorted(AUDIO_FOLDER.glob("**/*"))
audios = [a for a in audios if a.suffix.lower() in (".wav", ".mp3")]

if audios:
    SINGLE_AUDIO = audios[0]
    result = predict(SINGLE_AUDIO)

    print(f"\n🔊 Archivo    : {result['filename']}")
    print(f"🏆 Predicción : {result['prediction']}")
    print(f"📊 Confianza  : {result['confidence']:.2%}")
    print(f"\n📋 Top-{TOP_K}:")
    for rank, (label, prob) in enumerate(result["top_k"], 1):
        bar = "█" * int(prob * 30)
        print(f"  {rank}. {label:<25} {prob:.2%}  {bar}")
else:
    print(f"⚠️  No hay audios .wav/.mp3 en {AUDIO_FOLDER}")


🔊 Archivo    : 11325622-police-siren-sound-effect-240674.mp3
🏆 Predicción : siren_alarm
📊 Confianza  : 94.05%

📋 Top-5:
  1. siren_alarm               94.05%  ████████████████████████████
  2. traffic                   5.17%  █
  3. dog                       0.46%  
  4. fight                     0.10%  
  5. fire                      0.08%  


## 9 · Inferencia por lotes sobre toda la carpeta

In [42]:
from tqdm import tqdm  # en vez de from tqdm.notebook import tqdm

audios = sorted(AUDIO_FOLDER.glob("**/*"))
audios = [a for a in audios if a.suffix.lower() in (".wav", ".mp3")]

if not audios:
    raise FileNotFoundError(f"No se encontraron audios en {AUDIO_FOLDER}")

print(f"📂 {len(audios)} audios encontrados en {AUDIO_FOLDER}\n")

rows = []
errors = []

for audio_path in tqdm(audios, desc="Procesando audios"):
    try:
        result = predict(audio_path)
        row = {
            "filename"   : result["filename"],
            "prediction" : result["prediction"],
            "confidence" : result["confidence"],
        }
        # Añadir columna por cada clase del top-k
        for label, prob in result["top_k"]:
            row[f"prob_{label}"] = round(prob, 4)
        rows.append(row)
    except Exception as e:
        errors.append({"filename": audio_path.name, "error": str(e)})
        print(f"❌ Error en {audio_path.name}: {e}")

results_df = pd.DataFrame(rows)

print(f"\n✅ Procesados: {len(rows)}  |  Errores: {len(errors)}")
results_df.head(20)

📂 18 audios encontrados en /home/andres/Documentos/proyecto4geeks/tests/audios/alertables



Procesando audios: 100%|██████████| 18/18 [00:00<00:00, 44.08it/s]


✅ Procesados: 18  |  Errores: 0


,filename,prediction,confidence,prob_siren_alarm,prob_traffic,prob_dog,prob_fight,prob_fire,prob_crying,prob_gun_explosion,prob_construction_noise,prob_car_crash,prob_glass_breaking
0,11325622-police-siren-sound-effect-240674.mp3,siren_alarm,0.940471,0.9405,0.0517,0.0046,0.0010,0.0008,NaN,NaN,NaN,NaN,NaN
1,ElevenLabs_A_6_to_7-year-old_child_crying_and_...,crying,0.998963,NaN,0.0000,0.0001,0.0008,0.0000,0.9990,NaN,NaN,NaN,NaN
2,audio_607a0.mp3,gun_explosion,0.914862,NaN,0.0034,NaN,NaN,0.0676,NaN,0.9149,0.0094,0.0025,NaN
3,child-crime-aw2xrhhk.wav,fight,0.920817,0.0095,0.0098,0.0155,0.9208,NaN,0.0399,NaN,NaN,NaN,NaN
4,dragon-studio-car-crash-sound-effect-376874.mp3,gun_explosion,0.774338,NaN,0.0014,NaN,NaN,0.1436,NaN,0.7743,0.0032,NaN,0.0769
5,dragon-studio-dog-barking-406629.mp3,dog,1.000000,0.0000,0.0000,1.0000,0.0000,NaN,NaN,NaN,0.0000,NaN,NaN
6,explosion-meme_dTCfAHs.mp3,gun_explosion,0.992856,NaN,NaN,NaN,NaN,0.0040,NaN,0.9929,0.0003,0.0020,0.0006
7,freesound_community-car-crash-edit-two-92001.mp3,fire,0.298889,NaN,0.2966,0.0578,NaN,0.2989,NaN,0.1302,0.1138,NaN,NaN
8,freesound_community-dog-barking-70772.mp3,dog,1.000000,NaN,0.0000,1.0000,NaN,0.0000,NaN,0.0000,0.0000,NaN,NaN
9,freesound_community-glass-shatter-7-95202.mp3,glass_breaking,0.672802,0.0461,0.0473,NaN,NaN,0.0382,NaN,NaN,0.1852,NaN,0.6728


## 10 · Resumen de predicciones

In [43]:
if not results_df.empty:
    summary = (
        results_df
        .groupby("prediction")
        .agg(
            count=("filename", "count"),
            avg_confidence=("confidence", "mean"),
        )
        .sort_values("count", ascending=False)
        .reset_index()
    )
    summary["avg_confidence"] = summary["avg_confidence"].map("{:.2%}".format)
    print("📊 Distribución de predicciones:")
    display(summary)

    # Archivos con confianza baja (puede necesitar revisión)
    CONFIDENCE_THRESHOLD = 0.50
    low_conf = results_df[results_df["confidence"] < CONFIDENCE_THRESHOLD]
    if not low_conf.empty:
        print(f"\n⚠️  {len(low_conf)} audios con confianza < {CONFIDENCE_THRESHOLD:.0%}:")
        display(low_conf[["filename", "prediction", "confidence"]])

📊 Distribución de predicciones:


,prediction,count,avg_confidence
0,gun_explosion,5,91.85%
1,fire,3,55.38%
2,traffic,2,66.05%
3,glass_breaking,2,71.08%
4,dog,2,100.00%
5,construction_noise,1,39.80%
6,crying,1,99.90%
7,fight,1,92.08%
8,siren_alarm,1,94.05%



⚠️  3 audios con confianza < 50%:


,filename,prediction,confidence
7,freesound_community-car-crash-edit-two-92001.mp3,fire,0.298889
10,freesound_community-m4-assault-rifle-long-burs...,fire,0.466741
15,strategy-game-building-so-ih3jfxnx.wav,construction_noise,0.397959


## 11 · Exportar resultados a CSV (opcional)

In [44]:
OUTPUT_CSV = AUDIO_FOLDER / "predictions.csv"

if not results_df.empty:
    results_df.to_csv(OUTPUT_CSV, index=False)
    print(f"✅ Resultados guardados en: {OUTPUT_CSV}")
else:
    print("⚠️  No hay resultados para exportar.")

✅ Resultados guardados en: /home/andres/Documentos/proyecto4geeks/tests/audios/alertables/predictions.csv
